In [ ]:
# ! pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 3.5 MB/s eta 0:00:00


In [ ]:
# Prep script for Google Colab to download and prepare data

import pandas as pd
import logging
import warnings
import os
from google.colab import drive
import gdown

# Configure logging and suppress warnings
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
warnings.filterwarnings("ignore")

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)
target_dir = '/content/drive/My Drive/Advanced Data Analytic Techniques'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
logging.info(f"Working directory set to: {os.getcwd()}")

# Define file path and download if not exists
input_file_name = "final_dataset_for_modeling (2).csv"
input_path = os.path.join(target_dir, input_file_name)
input_file_id = "1XkuI9ZBd6HPLPk0U_Z5MGdYzfli2_Kgg"

if not os.path.exists(input_path):
    try:
        file_url = f"https://drive.google.com/uc?id={input_file_id}"
        gdown.download(file_url, input_path, quiet=False)
        logging.info(f"Downloaded {input_file_name} to {input_path}")
    except Exception as e:
        logging.error(f"Failed to download file from Google Drive: {e}")
        raise
else:
    logging.info(f"File {input_file_name} already exists at {input_path}")

# Load and save data locally (optional step for Colab)
try:
    df = pd.read_csv(input_path, encoding='utf-8-sig')
    logging.info(f"Loaded dataset with {len(df)} rows from {input_path}")
    # Optionally save to a local path in Colab for later use
    local_output_path = '/content/final_dataset_for_modeling (2).csv'
    df.to_csv(local_output_path, index=False, encoding='utf-8-sig')
    logging.info(f"Saved dataset to {local_output_path} for local use")
except Exception as e:
    logging.error(f"Failed to load or save dataset: {e}")
    raise
print(df.info())
df.head(3)

Mounted at /content/drive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36525 entries, 0 to 36524
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   text              36525 non-null  object
 1   processed_text    36525 non-null  object
 2   lemmatized_text   36524 non-null  object
 3   category          36525 non-null  object
 4   encoded_category  36525 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.4+ MB
None


,text,processed_text,lemmatized_text,category,encoded_category
0,كثيرا ما ارتبطت المصادر التاريخية في الأندلس خ...,كثيرا ما ارتبطت المصادر التاريخية في الأندلس خ...,ارتبطت المصادر التاريخيه الاندلس خاصه كتب التر...,original_abstract,1
1,يعد العامل الثقافي احد ابرز الاسباب التي يعزى ...,يعد العامل الثقافي احد ابرز الاسباب التي يعزى ...,يعد العامل الثقافي احد ابرز الاسباب يعزي سقوط ...,original_abstract,1
2,شكلت تلك الجهود والمساعي الرائدة التي قام بها ...,شكلت تلك الجهود والمساعي الرائدة التي قام بها ...,شكلت الجهود والمساعي الرائده قاده الثوره خلال ...,original_abstract,1


In [ ]:
# # ===============================================
# # 0. Mount Google Drive and set paths
# # ===============================================
# from google.colab import drive
# import os
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from imblearn.over_sampling import SMOTE
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sentence_transformers import SentenceTransformer

# drive.mount('/content/drive')

# BASE_DIR = "/content/drive/MyDrive/Advanced Data Analytic Techniques"
# BERT_FOLDER = os.path.join(BASE_DIR, "bert_embeddings")
# TFIDF_FOLDER = os.path.join(BASE_DIR, "tfidf_embeddings")

# os.makedirs(BERT_FOLDER, exist_ok=True)
# os.makedirs(TFIDF_FOLDER, exist_ok=True)
# print(f"Drive folders ready:\n  BERT  -> {BERT_FOLDER}\n  TF-IDF -> {TFIDF_FOLDER}")

# # ===============================================
# # Common data prep
# # ===============================================

# # TEXT_COL = 'lemmatized_text'
# TEXT_COL = 'processed_text'
# LABEL_COL = 'encoded_category'

# df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
# df[LABEL_COL] = df[LABEL_COL].astype(int)

# print(f"Using TEXT_COL = '{TEXT_COL}', LABEL_COL = '{LABEL_COL}'")
# print(f"Dataset: {len(df)} rows, class distribution = {dict(zip(*np.unique(df[LABEL_COL], return_counts=True)))}")

# # Split once (same split used for both pipelines)
# train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df[LABEL_COL], random_state=42)
# val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df[LABEL_COL], random_state=42)
# print(f"Split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# # ===============================================
# # 1. BERT pipeline (L2 + StandardScaler + SMOTE)
# # ===============================================
# print("\n--- BERT Pipeline ---")
# model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# def generate_embeddings(model, texts, l2_normalize=True):
#     embs = model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
#     if l2_normalize:
#         norms = np.linalg.norm(embs, axis=1, keepdims=True)
#         norms[norms == 0] = 1.0
#         embs = embs / norms
#     return embs

# print("Encoding embeddings with L2 normalization...")
# X_train_raw = generate_embeddings(model, train_df[TEXT_COL].tolist(), l2_normalize=True)
# y_train = train_df[LABEL_COL].values
# X_val_raw = generate_embeddings(model, val_df[TEXT_COL].tolist(), l2_normalize=True)
# y_val = val_df[LABEL_COL].values
# X_test_raw = generate_embeddings(model, test_df[TEXT_COL].tolist(), l2_normalize=True)
# y_test = test_df[LABEL_COL].values
# print(f"BERT embeddings: {X_train_raw.shape}")

# print("Standardizing (fit on train only)...")
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train_raw)
# X_val_scaled = scaler.transform(X_val_raw)
# X_test_scaled = scaler.transform(X_test_raw)

# print("Applying SMOTE after scaling...")
# unique, counts = np.unique(y_train, return_counts=True)
# minority_count = counts.min()
# k_safe = max(1, min(5, minority_count - 1))
# smote = SMOTE(random_state=42, k_neighbors=k_safe)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
# print("Before SMOTE:", dict(zip(unique, counts)))
# print("After  SMOTE:", dict(zip(*np.unique(y_train_smote, return_counts=True))))

# np.save(os.path.join(BERT_FOLDER, "X_train_bert.npy"), X_train_smote)
# np.save(os.path.join(BERT_FOLDER, "y_train_bert.npy"), y_train_smote)
# np.save(os.path.join(BERT_FOLDER, "X_val_bert.npy"), X_val_scaled)
# np.save(os.path.join(BERT_FOLDER, "y_val_bert.npy"), y_val)
# np.save(os.path.join(BERT_FOLDER, "X_test_bert.npy"), X_test_scaled)
# np.save(os.path.join(BERT_FOLDER, "y_test_bert.npy"), y_test)
# print(f"BERT data saved successfully to: {BERT_FOLDER}")

# # ===============================================
# # 2. TF-IDF pipeline (StandardScaler + SMOTE)
# # ===============================================
# print("\n--- TF-IDF Pipeline ---")
# tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
# X_all = tfidf.fit_transform(df[TEXT_COL])
# y_all = df[LABEL_COL].values

# train_X, temp_X, train_y, temp_y = train_test_split(X_all, y_all, test_size=0.30, stratify=y_all, random_state=42)
# val_X, test_X, val_y, test_y = train_test_split(temp_X, temp_y, test_size=0.50, stratify=temp_y, random_state=42)
# print(f"TF-IDF matrices -> Train: {train_X.shape}, Val: {val_X.shape}, Test: {test_X.shape}")

# print("Standardizing TF-IDF features (fit on train only)...")
# scaler_tfidf = StandardScaler(with_mean=False)
# train_X_scaled = scaler_tfidf.fit_transform(train_X)
# val_X_scaled = scaler_tfidf.transform(val_X)
# test_X_scaled = scaler_tfidf.transform(test_X)

# print("Applying SMOTE after scaling...")
# unique, counts = np.unique(train_y, return_counts=True)
# minority_count = counts.min()
# k_safe = max(1, min(5, minority_count - 1))
# smote = SMOTE(random_state=42, k_neighbors=k_safe)
# X_train_smote, y_train_smote = smote.fit_resample(train_X_scaled, train_y)
# print("Before SMOTE:", dict(zip(unique, counts)))
# print("After  SMOTE:", dict(zip(*np.unique(y_train_smote, return_counts=True))))

# np.save(os.path.join(TFIDF_FOLDER, "X_train_tfidf.npy"), X_train_smote)
# np.save(os.path.join(TFIDF_FOLDER, "y_train_tfidf.npy"), y_train_smote)
# np.save(os.path.join(TFIDF_FOLDER, "X_val_tfidf.npy"), val_X_scaled)
# np.save(os.path.join(TFIDF_FOLDER, "y_val_tfidf.npy"), val_y)
# np.save(os.path.join(TFIDF_FOLDER, "X_test_tfidf.npy"), test_X_scaled)
# np.save(os.path.join(TFIDF_FOLDER, "y_test_tfidf.npy"), test_y)
# print(f"TF-IDF data saved successfully to: {TFIDF_FOLDER}")

In [ ]:
# ===================================================
# 1. Load all datasets + neat summary
# ===================================================
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# ------------------- Paths -------------------
BERT_FOLDER  = "/content/drive/MyDrive/Advanced Data Analytic Techniques/bert_embeddings"
TFIDF_FOLDER = "/content/drive/MyDrive/Advanced Data Analytic Techniques/tfidf_embeddings"

# ------------------- Helpers -------------------
def load_npy_object(path):
    """Load .npy that may contain a 0-d object array (e.g. sparse matrix)."""
    arr = np.load(path, allow_pickle=True)
    if isinstance(arr, np.ndarray) and arr.shape == ():
        arr = arr.item()
    return arr

def _arr_info(name, arr):
    is_sparse = sp.issparse(arr)
    shape = arr.shape if hasattr(arr, "shape") else np.shape(arr)
    dtype = str(arr.dtype) if hasattr(arr, "dtype") else type(arr).__name__
    nnz = arr.nnz if is_sparse else None
    return {
        "Name": name,
        "Type": ("sparse " + arr.__class__.__name__) if is_sparse else "dense",
        "Shape": f"{shape}",
        "DType": dtype,
        "NNZ": nnz
    }

def summarize_loaded_sets(prefix, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\n{prefix} SUMMARY")
    print("="*70)
    rows = [
        _arr_info(f"{prefix}_X_train", X_train),
        _arr_info(f"{prefix}_y_train", y_train),
        _arr_info(f"{prefix}_X_val",   X_val),
        _arr_info(f"{prefix}_y_val",   y_val),
        _arr_info(f"{prefix}_X_test",  X_test),
        _arr_info(f"{prefix}_y_test",  y_test),
    ]
    df_info = pd.DataFrame(rows, columns=["Name","Type","Shape","DType","NNZ"])
    print(df_info.to_string(index=False))

    def _counts(y):
        y_np = np.asarray(y)
        vals, cnts = np.unique(y_np, return_counts=True)
        return dict(zip(vals.tolist(), cnts.tolist()))
    print("-"*70)
    print("Class distribution:")
    print(f"  train : {_counts(y_train)}")
    print(f"  val   : {_counts(y_val)}")
    print(f"  test  : {_counts(y_test)}")
    print("="*70)

# ------------------- Load BERT -------------------
print("Loading BERT data...")
X_train_bert_raw = np.load(os.path.join(BERT_FOLDER, "X_train_bert.npy"))
y_train_bert_raw = np.load(os.path.join(BERT_FOLDER, "y_train_bert.npy"))
X_val_bert       = np.load(os.path.join(BERT_FOLDER, "X_val_bert.npy"))
y_val_bert       = np.load(os.path.join(BERT_FOLDER, "y_val_bert.npy"))
X_test_bert      = np.load(os.path.join(BERT_FOLDER, "X_test_bert.npy"))
y_test_bert      = np.load(os.path.join(BERT_FOLDER, "y_test_bert.npy"))

# ------------------- Load TF-IDF (robust) -------------------
print("\nLoading TF-IDF data (robust)...")
X_train_tfidf_raw = load_npy_object(os.path.join(TFIDF_FOLDER, "X_train_tfidf.npy"))
y_train_tfidf_raw = load_npy_object(os.path.join(TFIDF_FOLDER, "y_train_tfidf.npy"))
X_val_tfidf       = load_npy_object(os.path.join(TFIDF_FOLDER, "X_val_tfidf.npy"))
y_val_tfidf       = load_npy_object(os.path.join(TFIDF_FOLDER, "y_val_tfidf.npy"))
X_test_tfidf      = load_npy_object(os.path.join(TFIDF_FOLDER, "X_test_tfidf.npy"))
y_test_tfidf      = load_npy_object(os.path.join(TFIDF_FOLDER, "y_test_tfidf.npy"))

# ------------------- Summaries -------------------
summarize_loaded_sets("BERT",  X_train_bert_raw,  y_train_bert_raw,
                      X_val_bert, y_val_bert, X_test_bert, y_test_bert)
summarize_loaded_sets("TFIDF", X_train_tfidf_raw, y_train_tfidf_raw,
                      X_val_tfidf, y_val_tfidf, X_test_tfidf, y_test_tfidf)

# ===================================================
# 2. SMOTE on TRAIN only + 50% stratified sample
# ===================================================
SMOTE_RANDOM_STATE = 42
SAMPLE_RATIO       = 0.5   # 50% of the balanced train set

smote = SMOTE(random_state=SMOTE_RANDOM_STATE)

# ---------- BERT ----------
X_train_bert, y_train_bert = smote.fit_resample(X_train_bert_raw, y_train_bert_raw)
X_train_bert_sample, _, y_train_bert_sample, _ = train_test_split(
    X_train_bert, y_train_bert,
    train_size=SAMPLE_RATIO,
    stratify=y_train_bert,
    random_state=SMOTE_RANDOM_STATE,
    shuffle=True
)

# ---------- TF-IDF ----------
X_train_tfidf, y_train_tfidf = smote.fit_resample(X_train_tfidf_raw, y_train_tfidf_raw)
X_train_tfidf_sample, _, y_train_tfidf_sample, _ = train_test_split(
    X_train_tfidf, y_train_tfidf,
    train_size=SAMPLE_RATIO,
    stratify=y_train_tfidf,
    random_state=SMOTE_RANDOM_STATE,
    shuffle=True
)

print("\nAfter SMOTE + 50% sampling")
print(f"BERT  balanced train : {X_train_bert.shape}  → sample : {X_train_bert_sample.shape}")
print(f"TFIDF balanced train : {X_train_tfidf.shape} → sample : {X_train_tfidf_sample.shape}")

# ----------- BERT SCALING -----------
scaler_bert = StandardScaler()
X_train_bert_sample_scaled = scaler_bert.fit_transform(X_train_bert_sample)
X_val_bert_scaled = scaler_bert.transform(X_val_bert)
X_test_bert_scaled = scaler_bert.transform(X_test_bert)

# ----------- TF-IDF SCALING -----------
scaler_tfidf = StandardScaler(with_mean=False) if sp.issparse(X_train_tfidf_sample) else StandardScaler()
X_train_tfidf_sample_scaled = scaler_tfidf.fit_transform(X_train_tfidf_sample)
X_val_tfidf_scaled = scaler_tfidf.transform(X_val_tfidf)
X_test_tfidf_scaled = scaler_tfidf.transform(X_test_tfidf)


Loading BERT data...

Loading TF-IDF data (robust)...

BERT SUMMARY
        Name  Type        Shape   DType  NNZ
BERT_X_train dense (46944, 384) float32 None
BERT_y_train dense     (46944,)   int64 None
  BERT_X_val dense  (5479, 384) float32 None
  BERT_y_val dense      (5479,)   int64 None
 BERT_X_test dense  (5479, 384) float32 None
 BERT_y_test dense      (5479,)   int64 None
----------------------------------------------------------------------
Class distribution:
  train : {0: 23472, 1: 23472}
  val   : {0: 5030, 1: 449}
  test  : {0: 5030, 1: 449}

TFIDF SUMMARY
         Name              Type          Shape   DType       NNZ
TFIDF_X_train sparse csr_matrix (46944, 10000) float64 4479328.0
TFIDF_y_train             dense       (46944,)   int64       NaN
  TFIDF_X_val sparse csr_matrix  (5479, 10000) float64  476737.0
  TFIDF_y_val             dense        (5479,)   int64       NaN
 TFIDF_X_test sparse csr_matrix  (5479, 10000) float64  483396.0
 TFIDF_y_test             dense   

# **PHASE 4 & 5: MODEL BUILDING & COMPREHENSIVE EVALUATION**

In [ ]:
# ===================================================
# (4000 SAMPLES + OPTIMIZED TUNING)
# ===================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import scipy.sparse as sp
import joblib
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix, roc_auc_score,
    roc_curve, auc, classification_report
)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import plotly.io as pio

# ===================================================
# SETUP
# ===================================================

# Create folders
os.makedirs("trained_models", exist_ok=True)
os.makedirs("confusion_matrices", exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")


Device: cpu



In [ ]:
# ===================================================
# HELPERS
# ===================================================

def improve_features(X_train, X_val, X_test):
    if sp.issparse(X_train):
        scaler = StandardScaler(with_mean=False)
    else:
        scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

def save_model(model, model_name):
    os.makedirs("trained_models", exist_ok=True)
    file_path = os.path.join("trained_models", f"{model_name}.joblib")
    joblib.dump(model, file_path)
    return file_path
def save_confusion_matrix_plot(y_true, y_pred, model_name, embedding_name):
    import os
    import numpy as np
    import plotly.graph_objects as go
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix

    # Base directory (adjust if needed)
    BASE_DIR = globals().get("BASE_DIR", "/content/drive/My Drive/Advanced Data Analytic Techniques")

    # Ensure save directory exists
    cm_dir = os.path.join(BASE_DIR, "confusion_matrices", str(model_name))
    os.makedirs(cm_dir, exist_ok=True)

    # File paths
    base_fname = f"CM_{model_name}_{embedding_name}"
    html_path = os.path.join(cm_dir, base_fname + ".html")
    png_path = os.path.join(cm_dir, base_fname + ".png")

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    labels_sorted = np.unique(np.concatenate([np.unique(y_true), np.unique(y_pred)]))
    x_labels = [str(l) for l in labels_sorted]
    y_labels = [str(l) for l in labels_sorted]

    # -------- Plotly interactive (HTML) --------
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=x_labels,
        y=y_labels,
        colorscale='Blues',
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 14, "color": "black"},  # <-- changed to black
        hoverongaps=False
    ))
    fig.update_layout(
        title=f'Confusion Matrix — {model_name} ({embedding_name})',
        xaxis_title="Predicted Label",
        yaxis_title="True Label",
        height=520,
        width=560,
        plot_bgcolor='white',
        font=dict(size=14)
    )

    # Save interactive HTML
    fig.write_html(html_path, include_plotlyjs='cdn', full_html=True)

    # -------- PNG via Matplotlib --------
    plt.figure(figsize=(5.6, 5.2), dpi=160)
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f'{model_name} ({embedding_name})')
    plt.colorbar()
    tick_marks = np.arange(len(labels_sorted))
    plt.xticks(tick_marks, x_labels, rotation=45, ha="right")
    plt.yticks(tick_marks, y_labels)

    # all numbers in black
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="black",  # <-- always black
                     fontsize=10)
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(png_path, bbox_inches="tight")
    plt.close()

    # -------- Show interactive only --------
    print(f"\nConfusion Matrix: {model_name} ({embedding_name})")
    fig.show()

    return cm


def stratified_sample(X, y, n_samples=4000):
    if len(y) <= n_samples:
        return X, y
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=n_samples/len(y), random_state=42)
    _, idx = next(splitter.split(X, y))
    return X[idx], y[idx]

print("="*80)
print("Helper functions loaded")
print("="*80 + "\n")

# ===================================================
# FEATURE SCALING
# ===================================================
X_train_bert_scaled, X_val_bert_scaled, X_test_bert_scaled = improve_features(
    X_train_bert, X_val_bert, X_test_bert
)
X_train_tfidf_scaled, X_val_tfidf_scaled, X_test_tfidf_scaled = improve_features(
    X_train_tfidf, X_val_tfidf, X_test_tfidf
)

# ===================================================
# SAMPLE 4000 FOR TRAINING (FASTER)
# ===================================================
X_train_bert_sample, y_train_bert_sample = stratified_sample(X_train_bert_scaled, y_train_bert, 4000)
X_train_tfidf_sample, y_train_tfidf_sample = stratified_sample(X_train_tfidf_scaled, y_train_tfidf, 4000)

print(f"Training samples: BERT {X_train_bert_sample.shape} | TF-IDF {X_train_tfidf_sample.shape}\n")


Helper functions loaded

Training samples: BERT (4000, 384) | TF-IDF (4000, 10000)



In [ ]:
# ===================================================
# TASK 4.1: BASELINE MODEL (Logistic Regression)
# ===================================================
# ===================================================
# TASK 4.1: BASELINE MODEL
# ===================================================

print("\n" + "="*80)
print("TASK 4.1: BASELINE MODEL (Logistic Regression)")
print("="*80 + "\n")

def train_baseline(X_train, y_train, X_val, y_val, X_test, y_test, name):

    print(f"\n[Baseline] Training Logistic Regression on {name} ...")

    # 1) Handle sparse inputs safely (convert to dense if needed)
    if sp.issparse(X_train):
        X_train = X_train.toarray()
    if sp.issparse(X_val):
        X_val = X_val.toarray()
    if sp.issparse(X_test):
        X_test = X_test.toarray()

    model = LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        solver='liblinear',
        C=1.0,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    proba_val = model.predict_proba(X_val)
    is_binary = (proba_val.shape[1] == 2)

    if is_binary:
        val_probs = proba_val[:, 1]
        best_thr, best_f1 = 0.5, -1.0
        for thr in np.linspace(0.05, 0.95, 19):
            preds_val = (val_probs >= thr).astype(int)
            f1_macro = f1_score(y_val, preds_val, average='macro')
            if f1_macro > best_f1:
                best_f1, best_thr = f1_macro, thr
    else:
        best_thr = 0.5

    proba_test = model.predict_proba(X_test)
    if is_binary:
        y_probs = proba_test[:, 1]
        y_pred = (y_probs >= best_thr).astype(int)
        roc_val = roc_auc_score(y_test, y_probs)
    else:
        y_pred = np.argmax(proba_test, axis=1)
        try:
            roc_val = roc_auc_score(y_test, proba_test, multi_class='ovr', average='weighted')
        except Exception:
            roc_val = np.nan

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    bacc = balanced_accuracy_score(y_test, y_pred)

    print(f"[Baseline:{name}] Acc: {acc*100:.2f}% | Precision: {prec*100:.2f}% | "
          f"Recall: {rec*100:.2f}% | F1 (weighted): {f1_w*100:.2f}% | "
          f"ROC-AUC: {0 if np.isnan(roc_val) else roc_val*100:.2f}%")


    save_confusion_matrix_plot(y_test, y_pred, " Logistic Regression ", name)
    return {
        'Model': 'Logistic Regression (Baseline)',
        'Embedding': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1_w,
        'ROC-AUC': roc_val
    }


baseline_bert = train_baseline(X_train_bert_sample, y_train_bert_sample, X_val_bert_scaled, y_val_bert, X_test_bert_scaled, y_test_bert, "BERT")
baseline_tfidf = train_baseline(X_train_tfidf_sample, y_train_tfidf_sample, X_val_tfidf_scaled, y_val_tfidf, X_test_tfidf_scaled, y_test_tfidf, "TF-IDF")



TASK 4.1: BASELINE MODEL (Logistic Regression)


[Baseline] Training Logistic Regression on BERT ...
[Baseline:BERT] Acc: 90.31% | Precision: 90.69% | Recall: 90.31% | F1 (weighted): 90.49% | ROC-AUC: 87.60%

Confusion Matrix:  Logistic Regression  (BERT)



[Baseline] Training Logistic Regression on TF-IDF ...
[Baseline:TF-IDF] Acc: 94.36% | Precision: 94.97% | Recall: 94.36% | F1 (weighted): 94.60% | ROC-AUC: 96.48%

Confusion Matrix:  Logistic Regression  (TF-IDF)


In [ ]:
# ===================================================
# TASK 4.1: BASELINE MODEL (Naive Bayes)
# ===================================================

print("\n" + "="*80)
print("TASK 4.1: BASELINE MODEL (Naive Bayes)")
print("="*80 + "\n")

def train_nb(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training Naive Bayes ({name})...")
    import numpy as np
    import scipy.sparse as sp
    from sklearn.naive_bayes import MultinomialNB, ComplementNB, GaussianNB
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

    # Detect problem type
    is_multiclass = len(np.unique(y_train)) > 2

    # Helper: check non-negativity (for MNB/CNB)
    def is_non_negative(X):
        if sp.issparse(X):
            return X.min() >= 0
        return np.min(X) >= 0

    # Branch: choose NB family & light tuning on validation
    best_model = None
    best_score = -1.0
    best_thr = 0.5  # used for binary

    # Case A: prefer MNB/CNB when features are non-negative (TF-IDF or similar)
    if (sp.issparse(X_train) and is_non_negative(X_train)) or (not sp.issparse(X_train) and is_non_negative(X_train)):
        # Keep sparse as-is (MNB/CNB support sparse); if dense, use as-is
        candidates = []
        for alpha in [0.1, 0.5, 1.0, 2.0]:
            candidates.append(MultinomialNB(alpha=alpha))
            candidates.append(ComplementNB(alpha=alpha))
        for model in candidates:
            model.fit(X_train, y_train)
            proba_val = model.predict_proba(X_val)
            if not is_multiclass:
                probs = proba_val[:, 1]
                best_thr_p, best_combo_p = 0.5, -1.0
                for thr in np.linspace(0.1, 0.9, 17):
                    preds = (probs >= thr).astype(int)
                    combo = (accuracy_score(y_val, preds) + f1_score(y_val, preds, average='weighted')) / 2
                    if combo > best_combo_p:
                        best_combo_p, best_thr_p = combo, thr
                score = best_combo_p
                thr_for_model = best_thr_p
            else:
                preds = np.argmax(proba_val, axis=1)
                score = f1_score(y_val, preds, average='weighted')
                thr_for_model = None
            if score > best_score:
                best_score = score
                best_model = model
                if not is_multiclass:
                    best_thr = float(thr_for_model)
    else:
        # Case B: dense with negatives (e.g., BERT) -> GaussianNB (requires dense)
        if sp.issparse(X_train):
            X_train = X_train.toarray()
            X_val = X_val.toarray()
            X_test = X_test.toarray()
        for vs in [1e-9, 1e-8, 1e-7]:
            model = GaussianNB(var_smoothing=vs)
            model.fit(X_train, y_train)
            proba_val = model.predict_proba(X_val)
            if not is_multiclass:
                probs = proba_val[:, 1]
                best_thr_p, best_combo_p = 0.5, -1.0
                for thr in np.linspace(0.1, 0.9, 17):
                    preds = (probs >= thr).astype(int)
                    combo = (accuracy_score(y_val, preds) + f1_score(y_val, preds, average='weighted')) / 2
                    if combo > best_combo_p:
                        best_combo_p, best_thr_p = combo, thr
                score = best_combo_p
                thr_for_model = best_thr_p
            else:
                preds = np.argmax(proba_val, axis=1)
                score = f1_score(y_val, preds, average='weighted')
                thr_for_model = None
            if score > best_score:
                best_score = score
                best_model = model
                if not is_multiclass:
                    best_thr = float(thr_for_model)

    # Fit final (best_model already fitted on train)
    # Predict on test
    proba_test = best_model.predict_proba(X_test)
    if not is_multiclass:
        y_proba = proba_test[:, 1]
        y_pred = (y_proba >= best_thr).astype(int)
        auc_score = roc_auc_score(y_test, y_proba)
    else:
        y_pred = np.argmax(proba_test, axis=1)
        try:
            auc_score = roc_auc_score(y_test, proba_test, multi_class='ovr', average='weighted')
        except Exception:
            auc_score = np.nan

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"Acc: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")

    save_confusion_matrix_plot(y_test, y_pred, "NaiveBayes", name)
    save_model(best_model, f"nb_{name}")

    return {
        'Model': 'Naive Bayes',
        'Embedding': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc_score
    }

nb_bert  = train_nb(X_train_bert_sample,  y_train_bert_sample,  X_val_bert_scaled,  y_val_bert,  X_test_bert_scaled,  y_test_bert,  "BERT")
nb_tfidf = train_nb(X_train_tfidf_sample, y_train_tfidf_sample, X_val_tfidf_scaled, y_val_tfidf, X_test_tfidf_scaled, y_test_tfidf, "TF-IDF")



TASK 4.1: BASELINE MODEL (Naive Bayes)

Training Naive Bayes (BERT)...
Acc: 56.85% | Precision: 88.60% | Recall: 56.85% | F1: 66.28% | ROC-AUC: 65.01%

Confusion Matrix: NaiveBayes (BERT)


Training Naive Bayes (TF-IDF)...
Acc: 90.02% | Precision: 92.13% | Recall: 90.02% | F1: 90.87% | ROC-AUC: 90.18%

Confusion Matrix: NaiveBayes (TF-IDF)


# **TASK 4.2: TRADITIONAL ML MODELS (SVM, RF, XGBoost)**

In [ ]:
# ===================================================
# TASK 4.2: TRADITIONAL ML MODELS
# ===================================================

print("\n" + "="*80)
print("TASK 4.2: TRADITIONAL ML MODELS (SVM)")
print("="*80 + "\n")

def train_svm(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training SVM ({name})")

    # Step 1: Train linear SVM on train data
    base_svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42, dual='auto')
    base_svm.fit(X_train, y_train)

    # Step 2: Calibrate on validation to enable predict_proba
    model = CalibratedClassifierCV(base_svm, method='sigmoid', cv='prefit')
    model.fit(X_val, y_val)

    # Step 3: Evaluate on test
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)

    try:
        if len(np.unique(y_test)) > 2:
            auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
        else:
            auc_score = roc_auc_score(y_test, y_proba[:, 1])
    except Exception:
        auc_score = np.nan

    print(f"Params: {{'C': 1.0, 'kernel': 'linear (approx)', 'calibration': 'sigmoid'}}")
    print(f"Acc: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")

    # Keep your exact saving workflow
    save_confusion_matrix_plot(y_test, y_pred, "SVM", name)
    save_model(model, f"svm_{name}")

    return {'Model': 'SVM', 'Embedding': name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc_score}

svm_bert = train_svm(X_train_bert_sample, y_train_bert_sample, X_val_bert_scaled, y_val_bert, X_test_bert_scaled, y_test_bert, "BERT")
svm_tfidf = train_svm(X_train_tfidf_sample, y_train_tfidf_sample, X_val_tfidf_scaled, y_val_tfidf, X_test_tfidf_scaled, y_test_tfidf, "TF-IDF")



TASK 4.2: TRADITIONAL ML MODELS (SVM)

Training SVM (BERT)
Params: {'C': 1.0, 'kernel': 'linear (approx)', 'calibration': 'sigmoid'}
Acc: 91.55% | Precision: 88.29% | Recall: 91.55% | F1: 88.91% | ROC-AUC: 86.99%

Confusion Matrix: SVM (BERT)


Training SVM (TF-IDF)
Params: {'C': 1.0, 'kernel': 'linear (approx)', 'calibration': 'sigmoid'}
Acc: 94.10% | Precision: 93.40% | Recall: 94.10% | F1: 93.47% | ROC-AUC: 95.08%

Confusion Matrix: SVM (TF-IDF)


In [ ]:
print("\n" + "="*80)
print("TASK 4.2: TRADITIONAL ML MODELS (RF)")
print("="*80 + "\n")
def train_rf(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training Random Forest ({name})...")
    import numpy as np
    import scipy.sparse as sp
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

    # Sparse → dense if needed
    if sp.issparse(X_train):
        X_train, X_val, X_test = X_train.toarray(), X_val.toarray(), X_test.toarray()

    # Candidate params (strong but still fast)
    candidates = [
        {"n_estimators": 400, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 600, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 500, "max_depth": 25,   "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt"},
        {"n_estimators": 500, "max_depth": 15,   "min_samples_split": 2, "min_samples_leaf": 2, "max_features": "sqrt"},
        {"n_estimators": 400, "max_depth": None, "min_samples_split": 5, "min_samples_leaf": 1, "max_features": 0.5},
    ]

    is_multiclass = (len(np.unique(y_train)) > 2)
    best_params = None
    best_score = -1.0
    best_thr = 0.5  # for binary threshold tuning

    # Lightweight validation selection
    for p in candidates:
        rf = RandomForestClassifier(
            n_estimators=p["n_estimators"],
            max_depth=p["max_depth"],
            min_samples_split=p["min_samples_split"],
            min_samples_leaf=p["min_samples_leaf"],
            max_features=p["max_features"],
            class_weight='balanced_subsample',
            bootstrap=True,
            n_jobs=-1,
            random_state=42,
            oob_score=False
        )
        rf.fit(X_train, y_train)

        if is_multiclass:
            y_val_pred = rf.predict(X_val)
            score = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)
            thr_for_p = None
        else:
            val_proba = rf.predict_proba(X_val)[:, 1]
            best_thr_p, best_combo_p = 0.5, -1.0
            for thr in np.linspace(0.05, 0.95, 19):
                preds = (val_proba >= thr).astype(int)
                combo = (accuracy_score(y_val, preds) + f1_score(y_val, preds, average='weighted')) / 2
                if combo > best_combo_p:
                    best_combo_p, best_thr_p = combo, thr
            score = best_combo_p
            thr_for_p = best_thr_p

        if score > best_score:
            best_score = score
            best_params = p.copy()
            if not is_multiclass:
                best_thr = float(thr_for_p)

    # Final model on TRAIN with best params
    model = RandomForestClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        max_features=best_params["max_features"],
        class_weight='balanced_subsample',
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict on TEST (with tuned threshold for binary)
    y_proba_full = model.predict_proba(X_test)
    if is_multiclass:
        y_pred = np.argmax(y_proba_full, axis=1)
        try:
            auc_score = roc_auc_score(y_test, y_proba_full, multi_class='ovr', average='weighted')
        except Exception:
            auc_score = np.nan
    else:
        y_proba = y_proba_full[:, 1]
        y_pred  = (y_proba >= best_thr).astype(int)
        auc_score = roc_auc_score(y_test, y_proba)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"Params: {best_params}")
    print(f"Acc: {acc*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")

    save_confusion_matrix_plot(y_test, y_pred, "RandomForest", name)
    save_model(model, f"rf_{name}")

    return {'Model': 'Random Forest', 'Embedding': name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc_score}

# Calls unchanged

rf_bert = train_rf(X_train_bert_sample, y_train_bert_sample, X_val_bert_scaled, y_val_bert, X_test_bert_scaled, y_test_bert, "BERT")
rf_tfidf = train_rf(X_train_tfidf_sample, y_train_tfidf_sample, X_val_tfidf_scaled, y_val_tfidf, X_test_tfidf_scaled, y_test_tfidf, "TF-IDF")



TASK 4.2: TRADITIONAL ML MODELS (RF)

Training Random Forest (BERT)...
Params: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5}
Acc: 91.82% | F1: 88.49% | ROC-AUC: 79.54%

Confusion Matrix: RandomForest (BERT)


Training Random Forest (TF-IDF)...
Params: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Acc: 93.90% | F1: 93.34% | ROC-AUC: 95.64%

Confusion Matrix: RandomForest (TF-IDF)


In [ ]:
# XGBoost
print("\n" + "="*80)
print("TASK 4.2: TRADITIONAL ML MODELS (XGBoost)")
print("="*80 + "\n")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier
import scipy.sparse as sp
def train_xgb(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training XGBoost ({name})...")

    import numpy as np
    import scipy.sparse as sp
    from xgboost import XGBClassifier
    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

    # Dense conversion if needed
    if sp.issparse(X_train):
        X_train = X_train.toarray()
        X_val   = X_val.toarray()
        X_test  = X_test.toarray()

    # GPU if available (optional)
    tree_method = "hist"
    predictor   = "auto"
    try:
        import torch
        if torch.cuda.is_available():
            tree_method = "gpu_hist"
            predictor   = "gpu_predictor"
    except Exception:
        pass

    # Binary vs multiclass
    is_multiclass = (len(np.unique(y_train)) > 2)

    # scale_pos_weight (binary)
    if not is_multiclass:
        pos = int((y_train == 1).sum())
        neg = int((y_train != 1).sum())
        scale_pos_weight = float(max(neg / max(pos, 1), 1.0))
    else:
        scale_pos_weight = 1.0

    # Fast stratified subset for tuning
    def stratified_subset(X, y, max_samples=3000, random_state=42):
        if len(y) <= max_samples:
            return X, y
        sss = StratifiedShuffleSplit(n_splits=1, test_size=max_samples/len(y), random_state=random_state)
        _, idx = next(sss.split(X, y))
        return X[idx], y[idx]

    X_sub, y_sub = stratified_subset(X_train, y_train, max_samples=3000)

    # Compact, strong candidates (including n_estimators)
    candidates = [
        {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.10, "subsample": 0.9, "colsample_bytree": 0.9},
        {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.10, "subsample": 0.9, "colsample_bytree": 0.9},
        {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.10, "subsample": 1.0, "colsample_bytree": 1.0},
        {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.08, "subsample": 0.8, "colsample_bytree": 0.8},
    ]

    best_params = None
    best_score  = -1.0
    best_thr    = 0.5

    # Mini-tuning on subset (no early stopping)
    for p in candidates:
        model = XGBClassifier(
            n_estimators=p["n_estimators"],
            max_depth=p["max_depth"],
            learning_rate=p["learning_rate"],
            subsample=p["subsample"],
            colsample_bytree=p["colsample_bytree"],
            reg_lambda=1.0,
            objective=("binary:logistic" if not is_multiclass else "multi:softprob"),
            num_class=(None if not is_multiclass else len(np.unique(y_train))),
            random_state=42,
            tree_method=tree_method,
            predictor=predictor,
            max_bin=256,
            n_jobs=-1,
            verbosity=0,
            eval_metric=("auc" if not is_multiclass else "mlogloss"),
            scale_pos_weight=(scale_pos_weight if not is_multiclass else 1.0)
        )

        model.fit(X_sub, y_sub)

        proba_val = model.predict_proba(X_val)
        if not is_multiclass:
            probs = proba_val[:, 1]
            best_thr_p, best_combo_p = 0.5, -1.0
            for thr in np.linspace(0.1, 0.9, 17):
                preds = (probs >= thr).astype(int)
                combo = (accuracy_score(y_val, preds) + f1_score(y_val, preds, average="weighted")) / 2
                if combo > best_combo_p:
                    best_combo_p, best_thr_p = combo, thr
            score = best_combo_p
        else:
            preds = np.argmax(proba_val, axis=1)
            score = f1_score(y_val, preds, average="weighted")

        if score > best_score:
            best_score = score
            best_params = p.copy()
            if not is_multiclass:
                best_thr = float(best_thr_p)

    # Final model on FULL train (single fit)
    final = XGBClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        learning_rate=best_params["learning_rate"],
        subsample=best_params["subsample"],
        colsample_bytree=best_params["colsample_bytree"],
        reg_lambda=1.0,
        objective=("binary:logistic" if not is_multiclass else "multi:softprob"),
        num_class=(None if not is_multiclass else len(np.unique(y_train))),
        random_state=42,
        tree_method=tree_method,
        predictor=predictor,
        max_bin=256,
        n_jobs=-1,
        verbosity=0,
        eval_metric=("auc" if not is_multiclass else "mlogloss"),
        scale_pos_weight=(scale_pos_weight if not is_multiclass else 1.0)
    )
    final.fit(X_train, y_train)

    # Test evaluation
    proba_test = final.predict_proba(X_test)
    if not is_multiclass:
        y_proba = proba_test[:, 1]
        y_pred  = (y_proba >= best_thr).astype(int)
        auc_score = roc_auc_score(y_test, y_proba)
    else:
        y_pred  = np.argmax(proba_test, axis=1)
        try:
            auc_score = roc_auc_score(y_test, proba_test, multi_class="ovr", average="weighted")
        except Exception:
            auc_score = np.nan

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="weighted", zero_division=0)

    print(f"Best Parameters: {best_params}")
    if not is_multiclass:
        print(f"Best Threshold: {best_thr:.3f}")
    print(f"Accuracy: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")

    save_confusion_matrix_plot(y_test, y_pred, "XGBoost", name)
    save_model(final, f"xgb_{name.lower()}")

    return {
        'Model': 'XGBoost',
        'Embedding': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc_score
    }


# Train the model using BERT and TF-IDF embeddings
xgb_bert = train_xgb(X_train_bert_sample, y_train_bert_sample, X_val_bert_scaled, y_val_bert, X_test_bert_scaled, y_test_bert, "BERT")
xgb_tfidf = train_xgb(X_train_tfidf_sample, y_train_tfidf_sample, X_val_tfidf_scaled, y_val_tfidf, X_test_tfidf_scaled, y_test_tfidf, "TF-IDF")


TASK 4.2: TRADITIONAL ML MODELS (XGBoost)

Training XGBoost (BERT)...
Best Parameters: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 1.0, 'colsample_bytree': 1.0}
Best Threshold: 0.900
Accuracy: 91.66% | Precision: 89.14% | Recall: 91.66% | F1: 89.66% | ROC-AUC: 80.96%

Confusion Matrix: XGBoost (BERT)


Training XGBoost (TF-IDF)...
Best Parameters: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.9, 'colsample_bytree': 0.9}
Best Threshold: 0.650
Accuracy: 95.11% | Precision: 94.94% | Recall: 95.11% | F1: 95.02% | ROC-AUC: 96.73%

Confusion Matrix: XGBoost (TF-IDF)


In [ ]:
# ===================================================
# TASK 4.3: DEEP LEARNING MODEL
# ===================================================

print("\n" + "="*80)
print("TASK 4.3: DEEP LEARNING MODEL (Feedforward Neural Network)")
print("="*80 + "\n")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# --- device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_ffnn(input_dim, hidden1=512, hidden2=256):
    # Wider + BatchNorm for stability; same API & 2 outputs for CE loss
    return nn.Sequential(
        nn.Linear(input_dim, hidden1),
        nn.BatchNorm1d(hidden1),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(hidden1, hidden2),
        nn.BatchNorm1d(hidden2),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(hidden2, 2)
    ).to(device)

def train_ffnn(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training FFNN ({name})...")

    # --- Dataloaders
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
        batch_size=128, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
        batch_size=256
    )
    test_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
        batch_size=256
    )

    model = build_ffnn(X_train.shape[1])

    # --- class weights (robust) ---
    n_classes = int(np.max(y_train)) + 1
    counts = np.bincount(y_train, minlength=n_classes)
    weights = counts.max() / np.clip(counts, 1, None)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_f1 = 0.0
    best_model_state = None
    best_thr = 0.5
    patience = 7
    patience_counter = 0
    max_epochs = 30
    clip_norm = 1.0

    for epoch in range(max_epochs):
        model.train()
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()
            epoch_loss += loss.item()

        # ---- validation + threshold tuning (binary) ----
        model.eval()
        val_probs = []
        val_y_list = []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(device)
                logits = model(batch_X)
                probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                val_probs.extend(probs)
                val_y_list.extend(batch_y.numpy())

        val_probs = np.array(val_probs)
        val_y_np  = np.array(val_y_list)

        # scan thresholds to maximize weighted F1
        best_thr_epoch, best_f1_epoch = 0.5, -1.0
        for thr in np.linspace(0.05, 0.95, 19):
            preds = (val_probs >= thr).astype(int)
            f1_val = f1_score(val_y_np, preds, average='weighted')
            if f1_val > best_f1_epoch:
                best_f1_epoch, best_thr_epoch = f1_val, thr

        scheduler.step(best_f1_epoch)

        if best_f1_epoch > best_val_f1:
            best_val_f1 = best_f1_epoch
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_thr = float(best_thr_epoch)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # ---- restore best state ----
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    model.eval()

    # ---- test inference ----
    test_probs = []
    with torch.no_grad():
        for batch_X, _ in test_loader:
            logits = model(batch_X.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            test_probs.extend(probs)
    test_probs = np.array(test_probs)

    # use tuned threshold
    y_pred = (test_probs >= best_thr).astype(int)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    auc_score = roc_auc_score(y_test, test_probs)

    # print(f"Acc: {acc*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {auc_score*100:.2f}%")
    print(f"Acc: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")

    print(f"Best Val F1: {best_val_f1*100:.2f}% | Tuned Threshold: {best_thr:.3f}")

    # saving (same hooks)
    save_confusion_matrix_plot(y_test, y_pred, "FFNN", name)
    torch.save(model.state_dict(), f"trained_models/ffnn_{name}.pt")
    print("Saved\n")

    return {'Model': 'FFNN (Deep Learning)', 'Embedding': name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1, 'ROC-AUC': auc_score}

# call unchanged
ffnn_bert = train_ffnn(X_train_bert_sample, y_train_bert_sample, X_val_bert_scaled, y_val_bert, X_test_bert_scaled, y_test_bert, "BERT")



TASK 4.3: DEEP LEARNING MODEL (Feedforward Neural Network)

Training FFNN (BERT)...
Acc: 91.60% | Precision: 90.58% | Recall: 91.60% | F1: 90.99% | ROC-AUC: 86.23%
Best Val F1: 90.74% | Tuned Threshold: 0.900

Confusion Matrix: FFNN (BERT)


Saved



In [ ]:
# ===================================================
# TASK 4.3: DEEP LEARNING MODEL (Simple DNN)
# ===================================================

print("\n" + "="*80)
print("TASK 4.3: DEEP LEARNING MODEL (Simple DNN)")
print("="*80 + "\n")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_simple_dnn(input_dim, hidden=256, dropout=0.2):
    """
    One hidden layer MLP for speed.
    Output = 2 logits for CrossEntropyLoss (binary as 0/1).
    """
    return nn.Sequential(
        nn.Linear(input_dim, hidden),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, 2)
    ).to(device)

def train_dnn_simple(X_train, y_train, X_val, y_val, X_test, y_test, name):
    print(f"Training Simple DNN ({name})...")

    # Dataloaders
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
        batch_size=128, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
        batch_size=256
    )
    test_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
        batch_size=256
    )

    model = build_simple_dnn(X_train.shape[1], hidden=256, dropout=0.2)

    # Class weights (robust for imbalance)
    n_classes = int(np.max(y_train)) + 1
    counts = np.bincount(y_train, minlength=n_classes)
    weights = counts.max() / np.clip(counts, 1, None)
    class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=5e-4)  # slightly higher lr for shallow net
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_f1 = 0.0
    best_model_state = None
    best_thr = 0.5
    patience = 5
    patience_counter = 0
    max_epochs = 18
    clip_norm = 1.0

    for epoch in range(max_epochs):
        # Train
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()

        # Validate and threshold tuning (binary)
        model.eval()
        val_probs, val_true = [], []
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                logits = model(batch_X.to(device))
                probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
                val_probs.extend(probs)
                val_true.extend(batch_y.numpy())

        val_probs = np.array(val_probs)
        val_true = np.array(val_true)

        best_thr_epoch, best_f1_epoch = 0.5, -1.0
        for thr in np.linspace(0.1, 0.9, 17):
            preds = (val_probs >= thr).astype(int)
            f1_val = f1_score(val_true, preds, average='weighted', zero_division=0)
            if f1_val > best_f1_epoch:
                best_f1_epoch, best_thr_epoch = f1_val, thr

        scheduler.step(best_f1_epoch)

        if best_f1_epoch > best_val_f1:
            best_val_f1 = best_f1_epoch
            best_thr = float(best_thr_epoch)
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # Restore best
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    model.eval()

    # Test
    test_probs = []
    with torch.no_grad():
        for batch_X, _ in test_loader:
            logits = model(batch_X.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            test_probs.extend(probs)
    test_probs = np.array(test_probs)
    y_pred = (test_probs >= best_thr).astype(int)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    auc_score = roc_auc_score(y_test, test_probs)

    print(f"Acc: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}% | ROC-AUC: {0 if np.isnan(auc_score) else auc_score*100:.2f}%")
    print(f"Best Val F1: {best_val_f1*100:.2f}% | Tuned Threshold: {best_thr:.3f}")

    # Save (same hooks)
    save_confusion_matrix_plot(y_test, y_pred, "SimpleDNN", name)
    torch.save(model.state_dict(), f"trained_models/simple_dnn_{name}.pt")

    return {
        'Model': 'Simple DNN',
        'Embedding': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc_score
    }

simple_dnn_bert = train_dnn_simple(
    X_train_bert_sample, y_train_bert_sample,
    X_val_bert_scaled,  y_val_bert,
    X_test_bert_scaled, y_test_bert,
    "BERT"
)



TASK 4.3: DEEP LEARNING MODEL (Simple DNN)

Training Simple DNN (BERT)...
Acc: 91.86% | Precision: 90.62% | Recall: 91.86% | F1: 91.07% | ROC-AUC: 87.13%
Best Val F1: 90.79% | Tuned Threshold: 0.900

Confusion Matrix: SimpleDNN (BERT)


Saved



In [ ]:
# ===================================================
# TASK 4.4: COMPREHENSIVE EVALUATION
# ===================================================

print("\n" + "="*80)
print("TASK 4.4: COMPREHENSIVE EVALUATION & RESULTS")
print("="*80 + "\n")

results = pd.DataFrame([
    baseline_bert, baseline_tfidf,
    nb_bert, nb_tfidf,
    svm_bert, svm_tfidf,
    rf_bert, rf_tfidf,
    xgb_bert, xgb_tfidf,
    ffnn_bert,
    simple_dnn_bert
])

results_excel = "all_models_evaluation.xlsx"
results.to_excel(results_excel, index=False)

print("="*80)
print("ALL MODELS COMPARISON")
print("="*80)
print(results.round(4).to_string(index=False))

best_idx = results['F1-Score'].idxmax()
best_model = results.iloc[best_idx]

print("\n" + "="*80)
print("BEST MODEL")
print("="*80)
print(f"Model: {best_model['Model']} ({best_model['Embedding']})")
print(f"F1-Score: {best_model['F1-Score']*100:.2f}%")
print(f"Accuracy: {best_model['Accuracy']*100:.2f}%")
print(f"ROC-AUC: {best_model['ROC-AUC']*100:.2f}%")
print("="*80)

print(f"\nConfusion matrices saved to: confusion_matrices/")
print(f"Models saved to: trained_models/")
print(f"Results saved to: {results_excel}")


TASK 4.4: COMPREHENSIVE EVALUATION & RESULTS

ALL MODELS COMPARISON
                         Model Embedding  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression (Baseline)      BERT    0.9031     0.9069  0.9031    0.9049   0.8760
Logistic Regression (Baseline)    TF-IDF    0.9436     0.9497  0.9436    0.9460   0.9648
                   Naive Bayes      BERT    0.5685     0.8860  0.5685    0.6628   0.6501
                   Naive Bayes    TF-IDF    0.9002     0.9213  0.9002    0.9087   0.9018
                           SVM      BERT    0.9155     0.8829  0.9155    0.8891   0.8699
                           SVM    TF-IDF    0.9410     0.9340  0.9410    0.9347   0.9508
                 Random Forest      BERT    0.8941     0.8895  0.8941    0.8918   0.7965
                 Random Forest    TF-IDF    0.9390     0.9319  0.9390    0.9334   0.9564
                       XGBoost      BERT    0.9166     0.8914  0.9166    0.8966   0.8096
                       XGBoost    TF-IDF 

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import os

# Sort by accuracy (descending)
results_sorted = results.sort_values("Accuracy", ascending=False)

# ====================================================
# 1) Scatter Plot (Precision vs Recall)
# ====================================================
fig1 = px.scatter(
    results_sorted,
    x="Precision", y="Recall",
    size="Accuracy",
    color="Model",
    text="Embedding",
    hover_data=["F1-Score", "ROC-AUC"],
    title="Model Performance Overview (Precision vs Recall)",
    template="plotly_white"
)
fig1.update_traces(textposition="top center")
fig1.show()

# ====================================================
# 2) Horizontal Bar Chart (Top 10 by Accuracy)
# ====================================================
top10 = results_sorted.head(10)
fig2 = px.bar(
    top10,
    x="Accuracy", y="Model",
    color="Embedding",
    orientation="h",
    text=top10["Accuracy"].apply(lambda v: f"{v*100:.2f}%"),
    title="Top 10 Models by Accuracy",
    template="simple_white"
)
fig2.update_traces(textfont_size=13, textposition="outside")
fig2.update_layout(height=550)
fig2.show()

# ====================================================
# 3) Radar Chart (Normalized Performance Comparison)
# ====================================================
norm_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
results_norm = results.copy()

# Normalize values between 0 and 1
for c in norm_cols:
    results_norm[c] = (results_norm[c] - results_norm[c].min()) / (results_norm[c].max() - results_norm[c].min())

radar_fig = go.Figure()
for _, row in results_norm.iterrows():
    radar_fig.add_trace(go.Scatterpolar(
        r=[row[c] for c in norm_cols],
        theta=norm_cols,
        fill='toself',
        name=f"{row['Model']} ({row['Embedding']})"
    ))

radar_fig.update_layout(
    title="Radar Chart: Model Comparison Across Metrics",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    height=600
)
radar_fig.show()

# ====================================================
# 4) Heatmap (Performance Summary)
# ====================================================
fig4 = px.imshow(
    results_sorted[norm_cols],
    labels=dict(x="Metric", y="Model", color="Score"),
    x=norm_cols,
    y=[f"{m} ({e})" for m, e in zip(results_sorted['Model'], results_sorted['Embedding'])],
    color_continuous_scale="Blues",
    title="Model Comparison Heatmap",
)
fig4.update_layout(height=600) # width=1000
fig4.show()

# ====================================================
# 5) Save all figures to Drive directory
# ====================================================
BASE_DIR = globals().get("BASE_DIR", "/content/drive/My Drive/Advanced Data Analytic Techniques")
results_dir = os.path.join(BASE_DIR, "results_visuals")
os.makedirs(results_dir, exist_ok=True)

fig1.write_html(os.path.join(results_dir, "Performance_Scatter.html"))
fig2.write_html(os.path.join(results_dir, "Top10_Bar.html"))
radar_fig.write_html(os.path.join(results_dir, "Radar_Chart.html"))
fig4.write_html(os.path.join(results_dir, "Heatmap.html"))
